In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import precision_score, recall_score, mean_absolute_error,mean_squared_error


In [2]:
#loading the Rating dataset
df=pd.read_csv(r"C:\Users\HP\Desktop\Bookify\Database\Cleaned_Datasets\Ratings_Cleaned.csv")

In [3]:
df.head()

,user_id,book_id,ratings
0,276726,0155061224,5
1,276729,052165615X,3
2,276729,0521795028,6
3,276736,3257224281,8
4,276737,0600570967,6


### 1. Prepare the Interaction Matrix

In [4]:
# Limit to the first 30 rows
df_100 = df.head(100)

In [5]:
# Create the user-item interaction matrix
interaction_matrix = df_100.pivot_table(index='user_id', columns='book_id', values='ratings',  fill_value=0)


print(f"Interaction matrix shape: {interaction_matrix.shape}")
interaction_matrix.head()

Interaction matrix shape: (38, 100)


book_id,0006379702,000651118X,0060096195,0060517794,0091830893,0140260498,0141310340,0142302198,0155061224,0156006065,...,8478442588,8478884831,8478885218,8478885463,8478886044,8484330478,8484332039,8879839993,9057868059,N3453124715
user_id,,,,,,,,,,,,,,,,,,,,,
276726,0,0,0,0,0,0,0,0,5,0,...,0,0,0,0,0,0,0,0,0,0
276729,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
276736,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
276737,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
276744,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### 2. User-User Collaborative Filtering

In [6]:
# Create a User-Item matrix
interaction_matrix_filled = interaction_matrix.fillna(0)  # Fill missing values with 0

# Prepare the feature matrix for KNNClassifier
X = interaction_matrix_filled.values

# Train a NearestNeighbors model
knn_model = NearestNeighbors(metric='cosine', n_neighbors=6)  
knn_model.fit(X)  


NearestNeighbors(metric='cosine', n_neighbors=6)

In [7]:
# Function to get User-User Recommendations
def user_user_knn_recommendations(user_id, top_n=5, k=5):
    user_idx = interaction_matrix_filled.index.get_loc(user_id)

    # Predict similar users using the classifier
    distances, indices = knn_model.kneighbors([X[user_idx]], n_neighbors=k+1)

    similar_users = interaction_matrix_filled.index[indices.flatten()[1:]]
    
    #  Collect books liked (rated > 3) by similar users
    recommended_books = []
    for sim_user in similar_users:
        liked_books = interaction_matrix.loc[sim_user][interaction_matrix.loc[sim_user] > 3].index.tolist()
        recommended_books.extend(liked_books)

    # Remove books the target user has already rated
    already_rated = interaction_matrix.loc[user_id][interaction_matrix.loc[user_id] > 0].index.tolist()
    final_books = [book for book in recommended_books if book not in already_rated]

    # If no books are left after filtering, return empty list
    if not final_books:
        return []

    # Step 4.5: Recommend top N books (most frequently liked)
    return pd.Series(final_books).value_counts().head(top_n).index.tolist()

# Test for a few users
for user in df['user_id'].unique()[:5]:
    print(f"User-User Recommendations for User {user}: {user_user_knn_recommendations(user)}")


User-User Recommendations for User 276726: ['8879839993', '0091830893', '0586207414', '0812571029', '8423996565']
User-User Recommendations for User 276729: ['0395547032', '8423996565', '8426449476', '8426449573', '8478884831']
User-User Recommendations for User 276736: ['8879839993', '0395547032', '8423996565', '8426449476', '8426449573']
User-User Recommendations for User 276737: ['8879839993', '0395547032', '8423996565', '8426449476', '8426449573']
User-User Recommendations for User 276744: ['0440414121', '8423996565', '8426449476', '8426449573', '8478884831']


### 3. Implement Item-Item Collaborative Filtering

In [8]:
# Compute item-item cosine similarity
item_similarity = pd.DataFrame(cosine_similarity(interaction_matrix.T),
                               index=interaction_matrix.columns,
                               columns=interaction_matrix.columns)


In [9]:
def item_item_recommendations(user_id, top_n=5, sim_n=5):
    
    # Get books rated > 3 by the user
    liked_books = interaction_matrix.loc[user_id][interaction_matrix.loc[user_id] > 3].index
    recommendations = []

    
    for book in liked_books:
        similar_books = item_similarity[book].sort_values(ascending=False)
        top_similar_books = similar_books.iloc[1:sim_n+1].index 
        for similar_book in top_similar_books:
            if similar_book not in liked_books:
                recommendations.append(similar_book)

    # Return top recommended books based on frequency
    return pd.Series(recommendations).value_counts().head(top_n).index.tolist()

# Test on 5 users
for user in interaction_matrix.index[:5]:
    print(f"User {user} => Item-Item Recommendations: {item_item_recommendations(user)}")

User 276726 => Item-Item Recommendations: ['0006379702', '3442422035', '3453213025', '3453137442', '3453092007']
User 276729 => Item-Item Recommendations: ['3442435773', '347354034X', '3453213025', '3453137442']
User 276736 => Item-Item Recommendations: ['3442422035', '3453213025', '3453137442', '3453092007', '3442449820']
User 276737 => Item-Item Recommendations: ['0006379702', '3442435773', '3453213025', '3453137442', '3453092007']
User 276744 => Item-Item Recommendations: ['0006379702', '3442435773', '3453213025', '3453137442', '3453092007']


### 4. Matrix Factorization

In [10]:
#Apply SVD to reduce dimensions (20 features)
svd = TruncatedSVD(n_components=20, random_state=42)
compressed_matrix = svd.fit_transform(interaction_matrix)

#Rebuild the matrix (predict missing ratings)
reconstructed_matrix = np.dot(compressed_matrix, svd.components_)
predicted_ratings_df = pd.DataFrame(reconstructed_matrix, 
                                    index=interaction_matrix.index, 
                                    columns=interaction_matrix.columns)


In [11]:
#Recommend books for a user based on predicted ratings
def svd_recommendations(user_id, top_n=5):
    # Books the user has already rated
    rated_books = interaction_matrix.loc[user_id][interaction_matrix.loc[user_id] > 0].index
    
    # Get predicted ratings for books the user hasn't rated
    predictions = predicted_ratings_df.loc[user_id].drop(rated_books)
    
    # Return top N recommended book IDs
    return predictions.sort_values(ascending=False).head(top_n).index.tolist()

for user in interaction_matrix.index[:5]:
    recommendations = svd_recommendations(user)
    print(f"User {user} => SVD Recommendations: {recommendations}")


User 276726 => SVD Recommendations: ['0440225825', '0140260498', '0684867621', '342310538', '8484332039']
User 276729 => SVD Recommendations: ['8440682697', '3442136644', '342310538', '0395547032', '3442131340']
User 276736 => SVD Recommendations: ['0684867621', '0440225825', '0140260498', '0440414121', '3125785006']
User 276737 => SVD Recommendations: ['0140260498', '342310538', '3442131340', '8440682697', '3548603203']
User 276744 => SVD Recommendations: ['0140260498', '8440682697', '342310538', '3442131340', '3548603203']


### 5. Evaluation 

In [12]:
def compare_models(user_id):
    print(f"\nRecommendations for User {user_id}")
    print("Books already rated:", list(interaction_matrix.loc[user_id][interaction_matrix.loc[user_id] > 0].index))
    print("User-User CF Recommendations:", user_user_knn_recommendations(user_id))
    print("Item-Item CF Recommendations:", item_item_recommendations(user_id))
    print("SVD Recommendations:", svd_recommendations(user_id))
    print()

# Evaluate a few users
for uid in interaction_matrix.index[:3]:
    compare_models(uid)



Recommendations for User 276726
Books already rated: ['0155061224']
User-User CF Recommendations: ['8879839993', '0091830893', '0586207414', '0812571029', '8423996565']
Item-Item CF Recommendations: ['0006379702', '3442422035', '3453213025', '3453137442', '3453092007']
SVD Recommendations: ['0440225825', '0140260498', '0684867621', '342310538', '8484332039']


Recommendations for User 276729
Books already rated: ['052165615X', '0521795028']
User-User CF Recommendations: ['0395547032', '8423996565', '8426449476', '8426449573', '8478884831']
Item-Item CF Recommendations: ['3442435773', '347354034X', '3453213025', '3453137442']
SVD Recommendations: ['8440682697', '3442136644', '342310538', '0395547032', '3442131340']


Recommendations for User 276736
Books already rated: ['3257224281']
User-User CF Recommendations: ['8879839993', '0395547032', '8423996565', '8426449476', '8426449573']
Item-Item CF Recommendations: ['3442422035', '3453213025', '3453137442', '3453092007', '3442449820']
SVD

### Evaluation 

In [13]:
#Precision and Recall

def precision_recall_at_k(user_id, recommended_books, k=5):
    # Get the books the user has already rated
    actual_books = interaction_matrix.loc[user_id][interaction_matrix.loc[user_id] > 0].index.tolist()

    # Calculate precision and recall
    recommended_books = recommended_books[:k]
    recommended_set = set(recommended_books)
    actual_set = set(actual_books)

    # Precision: How many recommended books are actually rated by the user
    precision = len(recommended_set & actual_set) / len(recommended_set) if recommended_set else 0

    # Recall: How many actual rated books are in the top k recommended books
    recall = len(recommended_set & actual_set) / len(actual_set) if actual_set else 0

    return precision, recall


In [19]:
# Print all user IDs present in the interaction matrix
print("Available User IDs in Interaction Matrix:", interaction_matrix.index.tolist())

# Example: Evaluate for User 276726 (or any user present in the interaction matrix)
user_id = 276726

# Check if user_id exists in the index of the interaction matrix
if user_id in interaction_matrix.index:
    # Get recommendations from all models
    recommended_books_user_user = user_user_knn_recommendations(user_id)
    precision, recall = precision_recall_at_k(user_id, recommended_books_user_user)
    print("\nGet recommendations from all models")
    print(f"Precision@5 for User {user_id}: {precision}")
    print(f"Recall@5 for User {user_id}: {recall}")

    # Evaluate Item-Item CF
    recommended_books_item_item = item_item_recommendations(user_id)
    precision, recall = precision_recall_at_k(user_id, recommended_books_item_item)
    print("\nEvaluate Item-Item CF")
    print(f"Precision@5 for User {user_id}: {precision}")
    print(f"Recall@5 for User {user_id}: {recall}")

    # Evaluate SVD
    recommended_books_svd = svd_recommendations(user_id)
    precision, recall = precision_recall_at_k(user_id, recommended_books_svd)
    print("\n Evaluate SVD")
    print(f"Precision@5 for User {user_id}: {precision}")
    print(f"Recall@5 for User {user_id}: {recall}")
else:
    print(f"User ID {user_id} is not present in the interaction matrix.")

Available User IDs in Interaction Matrix: [276726, 276729, 276736, 276737, 276744, 276745, 276747, 276748, 276751, 276754, 276755, 276760, 276762, 276768, 276772, 276774, 276780, 276786, 276788, 276796, 276798, 276800, 276804, 276808, 276811, 276812, 276813, 276814, 276820, 276822, 276827, 276828, 276830, 276832, 276835, 276837, 276842, 276847]

Get recommendations from all models
Precision@5 for User 276835: 0.0
Recall@5 for User 276835: 0.0

Evaluate Item-Item CF
Precision@5 for User 276835: 0.0
Recall@5 for User 276835: 0.0

 Evaluate SVD
Precision@5 for User 276835: 0.0
Recall@5 for User 276835: 0.0


In [15]:
#MAE for Matrix Factorization (SVD):

def compute_mae():
    actual = interaction_matrix.values
    predicted = reconstructed_matrix
    
    return mean_absolute_error(actual.flatten(), predicted.flatten())

mae = compute_mae()
print(f"Mean Absolute Error (MAE): {mae}")


Mean Absolute Error (MAE): 0.03830665126250514


In [16]:
# RMSE for Matrix Factorization (SVD):

def compute_rmse():
    actual = interaction_matrix.values
    predicted = reconstructed_matrix
    
    return np.sqrt(mean_squared_error(actual.flatten(), predicted.flatten()))

rmse = compute_rmse()
print(f"Root Mean Squared Error (RMSE): {rmse}")

Root Mean Squared Error (RMSE): 0.44695418908610374


In [17]:
# Coverage:

def compute_coverage():
    recommended_books = set()
    for user_id in interaction_matrix.index:
        recommendations = user_user_knn_recommendations(user_id)
        recommended_books.update(recommendations)
    
    return len(recommended_books) / len(interaction_matrix.columns)

coverage = compute_coverage()
print(f"Coverage: {coverage}")


Coverage: 0.16


In [18]:
# Diversity:

def compute_diversity():
    recommended_books = set()
    for user_id in interaction_matrix.index:
        recommendations = user_user_knn_recommendations(user_id)
        recommended_books.update(recommendations)
    
    book_matrix = interaction_matrix[recommended_books]
    similarity_matrix = cosine_similarity(book_matrix.T)
    
    # Calculate average diversity (inverse similarity)
    num_recommendations = len(recommended_books)
    diversity = 1 - (similarity_matrix.sum() / (num_recommendations * (num_recommendations - 1)))
    
    return diversity

diversity = compute_diversity()
print(f"Diversity: {diversity}")

Diversity: 0.8083333333333333


**Metrics:**
- Mean Absolute Error (MAE): The average of absolute errors between predicted ratings and actual ratings is 0.038, which is very small and indicates that the system's predicted ratings are relatively accurate on average.
****


- Root Mean Squared Error (RMSE): The RMSE of 0.447 suggests that the predictions have some error but are still in a reasonable range for recommendation systems.
****

- Coverage: The coverage of 0.16 suggests that only 16% of all possible books have been recommended across users. A low coverage could indicate that the system is only recommending a small subset of books, which may be a limitation of the current model.

****
- Diversity: A diversity score of 0.808 indicates that the system is recommending a diverse set of books, which is good for providing a variety of suggestions to users.

### Observations

#### User-User Collaborative Filtering (CF)
##### Strengths:
- Able to generate recommendations by finding similar users (e.g., users 276726, 276729, 276736).
- Worked reasonably well when users had overlapping ratings (similar books rated by different users).

##### Limitations:
- **Popularity bias:** Tended to recommend the same set of popular books (e.g., "0345443683", "043935806X") across different users.
- **Struggled with sparse users:** When users had rated very few or uncommon books, it failed to find good matches.
- **Less personalized** — many users received similar recommendations.

#### Item-Item Collaborative Filtering (CF)
##### Strengths:
- Provided more personalized recommendations compared to User-User CF.
- Worked well even when users had rated only 1–2 books.
- Greater diversity: Different users received different book recommendations.
- Consistently generated suggestions as long as at least one rating was available.

##### Limitations:
- Still somewhat dependent on the presence of enough rated items.
- If a rated book was very rare, finding similar items could still be challenging.

#### SVD (Singular Value Decomposition - Matrix Factorization)
##### Strengths:
- Delivered the broadest and most diverse recommendations across all users.
- Handled sparse users effectively — even users with just one rating received good recommendations.
- Uncovered latent patterns — suggested books beyond direct similarities (not just by author or genre).
- Reduced repetition compared to User-User and Item-Item CF (better variety).

##### Limitations:
- Requires more computational resources to train compared to basic similarity-based methods.
- Slightly less intuitive — harder to "explain" exactly why a book was recommended without interpreting latent factors.

#### Specific Insights
- User-User CF showed strong popularity bias and needed overlapping user history to perform well.
- Item-Item CF was more stable, needing only one rated book and working better for cold-start users.
- SVD handled sparsity perfectly and offered the most diverse set of recommendations by learning hidden relationships in the data.